# 91 Charge / discharge LODES

$8\,\mathrm{h}$ / $100\,\mathrm{h}$ / $300\,\mathrm{h}$ thickness; asymmetric $L_\mathrm{c}$ vs $L_\mathrm{d}$.


## Governing equations

Default electrolyte $6\,\mathrm{M}$ KOH at $303\,\mathrm{K}$ unless noted.
Quantities use Jupyter MathJax with $\mathrm{}$ SI (siunitx is not available).
Patents: US12308414B2, EP4602674A1.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

here = Path.cwd()
for cand in (here, here.parent):
    if (cand / "src" / "ironair").is_dir() and (cand / "plant_sim").is_dir():
        sys.path.insert(0, str(cand))
        sys.path.insert(0, str(cand / "src"))
        break

from plant_sim.bootstrap import setup_path

setup_path()
from plant_sim.params import CAPACITY_FE_OH2_MAH_G, CAPACITY_MAGNETITE_MAH_G, default_params

In [ ]:
from IPython import get_ipython

_ipy = get_ipython()
if _ipy is not None:
    _ipy.run_line_magic("matplotlib", "inline")

sns.set(
    rc={
        "axes.labelsize": 12,
        "axes.titlesize": 18,
        "figure.figsize": (11, 8.5),
        "figure.dpi": 300,
        "figure.facecolor": "w",
        "figure.edgecolor": "k",
    }
)
P = default_params()
print("KOH default", P.c_KOH_mol_m3 / 1000, "M; T_ep", P.T_ep_sim_K, "K")
print("Faraday 960/320 checks", round(CAPACITY_FE_OH2_MAH_G, 2), round(CAPACITY_MAGNETITE_MAH_G, 2))

## Simulation

In [ ]:
from plant_sim.plant import PlantModel
from plant_sim.scenarios.mission import charge_discharge_inputs

for hrs, span in [(8, 1200), (100, 1200)]:
    pp = P.with_duration(hrs, hrs)
    plant = PlantModel(pp)
    r = plant.simulate((0, span), inputs=charge_discharge_inputs(span / 3, span / 3, pp), n_eval=100, rtol=1e-4)
    plot_frame = pd.DataFrame(
        {
            "t": (r["t"] / 60),
            "I": (r["alg"]["I_cell_A"]),
            "V": (r["alg"]["V_cell_V"]),
            "SOC": (r["alg"]["SOC"]),
        }
    )
    plot_long = plot_frame.melt(id_vars="t", var_name="quantity", value_name="value")
    fig, ax = plt.subplots()
    sns.lineplot(data=plot_long, x="t", y="value", hue="quantity", ax=ax, linewidth=2.0)
    ax.set_xlabel("t (min)")
    ax.set_ylabel("I/V/SOC")
    ax.set_title(f"{hrs} h thickness case")
    ax.text(0.0, -0.22, "US LODES duration scaling", transform=ax.transAxes, fontsize=8, va="top")
    fig.tight_layout()
    fig.subplots_adjust(bottom=0.22)
    print(hrs, "h L_anode_cm", round(pp.L_anode_m * 100, 2), "finite", np.all(np.isfinite(r["y"])))
pp = P.with_duration(100, 300)
print("asymmetric 100/300 h Lc,Ld cm", pp.L_charge_m * 100, pp.L_discharge_m * 100)

## Verification against patents / SSS002

In [ ]:
print("finite trajectories and patent-window parameters: see printed checks above")